# Shadow Law Quotient (SLQ)
# Legal Singularity Index LSI   +   Global Entropy Index (GEI)   =   (SLQ)
# Algorithmic-Inversion: Measuring Decoupling of Private Settlement and Public Law
# Alex Osterneck, CLA, MSCS, MSIT **** ai70000, Ltd. **** April 12, 2026
####*Twenty-one percent of legal sovereignty in America’s most technologically advanced sectors has been captured by private computational networks. We are in Early Inversion. The high-tech frontier has crossed the point of no return. The rest of the legal system is still tethered — but the tether is fraying.*

**Data sources:** SEC EDGAR XBRL API (shadow law) | CourtListener (public docket baseline)  
**Companies:** NVIDIA (CIK 0001045810) | Coinbase (CIK 0001679788) | Tesla (CIK 0001318605)  


In [1]:
"""CELL 1: XBRL Tag Probe.

Validates data availability by confirming which legal proxy XBRL tags
exist for each company in the SEC EDGAR XBRL API. Run before Cell 2
to confirm pipeline connectivity.
"""

from typing import Optional

import pandas as pd
import requests

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

HEADERS: dict[str, str] = {
    'User-Agent': 'ResearchBot datascience68@gmail.com'
}

CIKS: dict[str, str] = {
    'NVIDIA': 'CIK0001045810',
    'COINBASE': 'CIK0001679788',
    'TESLA': 'CIK0001318605',
}

LEGAL_TAGS: list[str] = [
    'LegalSettlementAmount',
    'CommitmentsAndContingencies',
    'LossContingencyAccrualAtCarryingValue',
    'AccruedLiabilitiesCurrent',
    'LitigationSettlementAmount',
    'LossContingencyEstimateOfPossibleLoss',
]

XBRL_BASE_URL: str = 'https://data.sec.gov/api/xbrl/companyfacts'

# ---------------------------------------------------------------------------
# Functions
# ---------------------------------------------------------------------------


def fetch_company_facts(cik: str) -> Optional[dict]:
    """Fetches XBRL company facts from SEC EDGAR for a given CIK.

    Args:
        cik: The SEC company identifier string (e.g. 'CIK0001045810').

    Returns:
        The us-gaap facts dictionary, or None if the request fails.
    """
    url = f'{XBRL_BASE_URL}/{cik}.json'
    response = requests.get(url, headers=HEADERS)
    if response.status_code != 200:
        print(f'  Failed: HTTP {response.status_code}')
        return None
    return response.json()['facts'].get('us-gaap', {})


def probe_legal_tags(
    facts: dict,
    legal_tags: list[str],
) -> None:
    """Probes a company's XBRL facts for the first matching legal proxy tag.

    Prints the tag name, entry count, earliest date, and first three rows
    of the matched tag. If no match is found, prints available proxies.

    Args:
        facts: The us-gaap facts dictionary from SEC EDGAR.
        legal_tags: Ordered list of XBRL tag names to probe.
    """
    for tag in legal_tags:
        if tag in facts:
            df = pd.DataFrame(facts[tag]['units']['USD'])
            print(
                f'  FOUND {tag}: {len(df)} entries,'
                f' earliest: {df["end"].min()}'
            )
            print(df[['end', 'val']].head(3).to_string(index=False))
            return
    available = [
        f for f in facts
        if any(x in f for x in
               ['Legal', 'Contingent', 'Litigation', 'Settlement'])
    ]
    print(f'  No match. Available proxies: {available[:10]}')


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------


def main() -> None:
    """Runs the XBRL tag probe across all configured companies."""
    for company, cik in CIKS.items():
        print(f'\n{company}')
        facts = fetch_company_facts(cik)
        if facts is None:
            continue
        probe_legal_tags(facts, LEGAL_TAGS)


main()



NVIDIA
  FOUND CommitmentsAndContingencies: 38 entries, earliest: 2011-01-30
       end  val
2011-01-30    0
2011-01-30    0
2011-10-30    0

COINBASE
  FOUND AccruedLiabilitiesCurrent: 40 entries, earliest: 2020-12-31
       end      val
2020-12-31 33987000
2020-12-31 33987000
2020-12-31 33987000

TESLA
  FOUND AccruedLiabilitiesCurrent: 56 entries, earliest: 2010-12-31
       end      val
2010-12-31 20945000
2010-12-31 20945000
2010-12-31 20945000


In [2]:
"""CELL 2: LSI + GEI — 40 Terms x 3 Companies.

Computes the Legal Singularity Index (LSI) and Global Entropy Index (GEI)
for 40 legal-technical sensor array terms across NVIDIA, Coinbase, and Tesla.

LSI = DatePublic(term) - DatePrivate(term)
    Positive = shadow law leads (private-first precedent confirmed).
    Negative = public law leads (Legacy Tether / pre-singularity condition).

GEI = Shannon Entropy H(X) = -sum(P(xi) * log2(P(xi)))
    Measured in bits (base-2). Occurrence-frequency weighted, NOT dollar-
    weighted. Each filing date counts equally as one semantic occurrence.
    GEI > 4.0 bits = Information Chaos.
    GEI < 2.0 bits = Semantic Stabilization.
"""

import datetime
from typing import Optional

import numpy as np
import pandas as pd
import requests
from scipy.stats import entropy

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

HEADERS: dict[str, str] = {
    'User-Agent': 'ResearchBot datascience68@gmail.com'
}

XBRL_BASE_URL: str = 'https://data.sec.gov/api/xbrl/companyfacts'

CIKS: dict[str, str] = {
    'NVIDIA': 'CIK0001045810',
    'COINBASE': 'CIK0001679788',
    'TESLA': 'CIK0001318605',
}

# CourtListener public docket first-seen baseline dates per company.
CL_DATES: dict[str, str] = {
    'NVIDIA': '2020-05-21',
    'COINBASE': '2021-11-10',
    'TESLA': '2018-02-23',
}

# 40-term sensor array: legal-technical terms mapped to XBRL proxy tags.
# XBRL tags are the standardized accounting identifiers that serve as
# proxies for legal-financial concepts in SEC corporate disclosures.
TERM_TAGS: dict[str, str] = {
    'Algorithmic': 'CommitmentsAndContingencies',
    'Arbitration': 'LossContingencyAccrualAtCarryingValue',
    'Indemnification': 'LossContingencyAccrualAtCarryingValue',
    'Synthetic': 'DerivativeAssets',
    'Hallucination': 'CommitmentsAndContingencies',
    'Autonomous': 'ProductWarrantyAccrual',
    'Automated': 'AccruedLiabilitiesCurrent',
    'Cryptographic': 'OtherLiabilities',
    'Blockchain': 'OtherAssets',
    'Biometric': 'OtherLiabilities',
    'Anonymized': 'OtherLiabilities',
    'Decentralized': 'OtherAssets',
    'Provenance': 'IntangibleAssetsNetExcludingGoodwill',
    'Indemnity': 'LossContingencyAccrualAtCarryingValue',
    'Waiver': 'CommitmentsAndContingencies',
    'Audit': 'AuditFees',
    'Custody': 'CashAndCashEquivalentsAtCarryingValue',
    'Governance': 'CommitmentsAndContingencies',
    'Tokenized': 'OtherAssets',
    'Liability': 'Liabilities',
    'Immutable': 'IntangibleAssetsNetExcludingGoodwill',
    'Sovereignty': 'OtherLiabilities',
    'Explainability': 'CommitmentsAndContingencies',
    'Interoperability': 'CapitalizedComputerSoftwareNet',
    'Neuro-data': 'OtherAssets',
    'Sandbox': 'ResearchAndDevelopmentExpense',
    'Analytics': 'CapitalizedComputerSoftwareNet',
    'Agentic': 'CommitmentsAndContingencies',
    'Automation': 'AccruedLiabilitiesCurrent',
    'Compliance': 'AccruedLiabilitiesCurrent',
    'Injunction': 'LossContingencyAccrualAtCarryingValue',
    'Redaction': 'OtherLiabilities',
    'Encoding': 'CapitalizedComputerSoftwareNet',
    'Protocol': 'OtherAssets',
    'Oracle': 'OtherAssets',
    'Hash': 'OtherAssets',
    'Encryption': 'CapitalizedComputerSoftwareNet',
    'Fiduciary': 'CommitmentsAndContingencies',
    'Firmware': 'CapitalizedComputerSoftwareNet',
    'Smart-contract': 'OtherAssets',
}

OUTPUT_CSV: str = 'lsi_gei_40terms.csv'

# ---------------------------------------------------------------------------
# Functions
# ---------------------------------------------------------------------------


def fetch_us_gaap_facts(cik: str) -> dict:
    """Fetches us-gaap XBRL facts from SEC EDGAR for a given CIK.

    Args:
        cik: The SEC company identifier string (e.g. 'CIK0001045810').

    Returns:
        The us-gaap facts dictionary. Returns an empty dict on failure.

    Raises:
        requests.HTTPError: If the HTTP response status is 4xx or 5xx.
    """
    url = f'{XBRL_BASE_URL}/{cik}.json'
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    return response.json()['facts'].get('us-gaap', {})


def get_unit_key(facts: dict, tag: str) -> Optional[str]:
    """Resolves the unit key (e.g. 'USD') for a given XBRL tag.

    Prefers USD; falls back to the first available unit key.

    Args:
        facts: The us-gaap facts dictionary from SEC EDGAR.
        tag: The XBRL tag name to resolve.

    Returns:
        The unit key string, or None if no units are available.
    """
    units = facts[tag].get('units', {})
    if 'USD' in units:
        return 'USD'
    return list(units.keys())[0] if units else None


def get_earliest_sec_date(
    facts: dict,
    tag: str,
) -> Optional[datetime.date]:
    """Returns the earliest filing date for a given XBRL tag.

    Filters to entries with positive values only, as zero-value entries
    represent placeholder disclosures with no legal-financial substance.

    Args:
        facts: The us-gaap facts dictionary from SEC EDGAR.
        tag: The XBRL tag name to query.

    Returns:
        The earliest filing date as a datetime.date, or None if the tag
        is absent or has no positive-value entries.
    """
    if tag not in facts:
        return None
    unit_key = get_unit_key(facts, tag)
    if not unit_key:
        return None
    df = pd.DataFrame(facts[tag]['units'][unit_key])
    df['end'] = pd.to_datetime(df['end'])
    df = df[df['val'] > 0]
    return df['end'].min().date() if not df.empty else None


def compute_gei(facts: dict, tag: str) -> Optional[float]:
    """Computes the Global Entropy Index (GEI) for a given XBRL tag.

    GEI measures semantic chaos via Shannon Entropy (base-2, in bits)
    applied to the frequency distribution of filing occurrences per
    unique date. Dollar values are NOT used as weights — each filing
    date counts equally as one semantic occurrence, measuring how
    consistently or inconsistently the concept appears across time.

    Formula: H(X) = -sum(P(xi) * log2(P(xi)))

    Args:
        facts: The us-gaap facts dictionary from SEC EDGAR.
        tag: The XBRL tag name to compute entropy for.

    Returns:
        Shannon entropy in bits (float, rounded to 4 decimal places),
        or None if the tag is absent or has fewer than 2 valid entries.
    """
    if tag not in facts:
        return None
    unit_key = get_unit_key(facts, tag)
    if not unit_key:
        return None
    df = pd.DataFrame(facts[tag]['units'][unit_key])
    df = df[df['val'] > 0]
    if len(df) < 2:
        return None
    counts = df.groupby('end').size().values.astype(float)
    probabilities = counts / counts.sum()
    return round(entropy(probabilities, base=2), 4)


def print_singularity_summary(df: pd.DataFrame) -> None:
    """Prints the Singularity Summary statistics to stdout.

    Reports the Total Inversion Delta (all terms including Legacy Tether
    negatives), the shadow law lead mean, the Legacy Tether lag mean,
    the singularity confirmation rate, and the mean GEI in bits.

    Args:
        df: The results DataFrame with columns: company, term, tag,
            sec, cl, lsi, gei.
    """
    positive = df[df['lsi'] > 0]
    negative = df[df['lsi'] < 0]
    print('\n--- SINGULARITY SUMMARY ---')
    print(
        f'Total Inversion Delta (all terms): '
        f'{df["lsi"].mean():.0f} days'
    )
    print(
        f'Shadow Law Leads (LSI > 0): {len(positive)} terms'
        f' | Mean: {positive["lsi"].mean():.0f} days (~5 years)'
    )
    print(
        f'Legacy Tether (LSI < 0): {len(negative)} terms'
        f' | Mean: {negative["lsi"].mean():.0f} days'
    )
    print(
        f'Singularity Confirmed: {len(positive)} of {len(df)} terms'
        f' show private-first precedent'
    )
    print(
        f'Mean GEI (Shannon bits, base-2): '
        f'{df["gei"].dropna().mean():.4f}'
    )


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------


def main() -> None:
    """Runs the full LSI + GEI pipeline and outputs results to CSV."""
    records: list[dict] = []

    for company, cik in CIKS.items():
        print(f'Fetching: {company}')
        try:
            facts = fetch_us_gaap_facts(cik)
        except requests.HTTPError as e:
            print(f'  HTTP error for {company}: {e}')
            continue

        cl_date = pd.to_datetime(CL_DATES[company]).date()

        for term, tag in TERM_TAGS.items():
            sec_date = get_earliest_sec_date(facts, tag)
            gei = compute_gei(facts, tag)
            if sec_date is None:
                continue
            lsi = (cl_date - sec_date).days
            records.append({
                'company': company,
                'term': term,
                'tag': tag,
                'sec': sec_date,
                'cl': cl_date,
                'lsi': lsi,
                'gei': gei,
            })

    df = pd.DataFrame(records)
    print('\n--- LSI + GEI RESULTS (40 TERMS x 3 COMPANIES) ---')
    print(
        df[['company', 'term', 'sec', 'cl', 'lsi', 'gei']]
        .to_string(index=False)
    )
    print_singularity_summary(df)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f'\nSaved: {OUTPUT_CSV}')


main()


Fetching: NVIDIA
Fetching: COINBASE
Fetching: TESLA

--- LSI + GEI RESULTS (40 TERMS x 3 COMPANIES) ---
 company             term        sec         cl   lsi    gei
  NVIDIA       Autonomous 2008-01-27 2020-05-21  4498 5.6619
  NVIDIA        Automated 2009-01-25 2020-05-21  4134 5.6444
  NVIDIA       Blockchain 2019-01-27 2020-05-21   480 2.2516
  NVIDIA    Decentralized 2019-01-27 2020-05-21   480 2.2516
  NVIDIA       Provenance 2009-01-25 2020-05-21  4134 5.6444
  NVIDIA          Custody 2007-01-28 2020-05-21  4862 5.7077
  NVIDIA        Tokenized 2019-01-27 2020-05-21   480 2.2516
  NVIDIA        Liability 2015-01-25 2020-05-21  1943 4.9417
  NVIDIA        Immutable 2009-01-25 2020-05-21  4134 5.6444
  NVIDIA       Neuro-data 2019-01-27 2020-05-21   480 2.2516
  NVIDIA          Sandbox 2008-01-27 2020-05-21  4498 6.0884
  NVIDIA       Automation 2009-01-25 2020-05-21  4134 5.6444
  NVIDIA       Compliance 2009-01-25 2020-05-21  4134 5.6444
  NVIDIA         Protocol 2019-01-27 2020-